# tamuno-sqlgen Python Tutorial

**A SQL template engine for Python** — write SQL templates with typed variables and optional sections, generate type-safe Python code, and execute queries against any database.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/klondenberg-bioptimus/tamuno-sqlgen/blob/master/Python/tamuno_sqlgen_tutorial.ipynb)

This notebook covers:
1. **Installation & Getting Started** — installing from GitHub, basic imports
2. **Basic Examples** — ad-hoc queries, code generation
3. **Escaping Special Characters** — using `\\` to produce literal `$`, `#`, `@`, `?`, `[`, `]`, `{`, `}`
4. **Advanced SQL Examples** — JOINs, aggregations, pagination, dynamic SQL
5. **Pandas Examples** — DataFrames, grouping, export
6. **Type-safe API Examples** — generated dataclasses, row types, introspection
7. **SQLAlchemy Examples** — engine usage, transactions, service layer pattern

---
## 1. Installation & Getting Started

Install `tamuno-sqlgen` directly from GitHub. The Python package lives in the `Python/` subdirectory of the monorepo.

In [ ]:
# Install tamuno-sqlgen from GitHub (Python subdirectory)
!pip install -q "tamuno-sqlgen @ git+https://github.com/klondenberg-bioptimus/tamuno-sqlgen.git#subdirectory=Python"

# Also install SQLAlchemy for Section 7
!pip install -q sqlalchemy

In [ ]:
# Verify the installation
from tamuno_sqlgen import (
    SQLGenApi,
    PythonCodeGenerator,
    SQLDialect,
    MySQLDialect,
    QueryFactory,
    TYPE_MAP,
)

print("tamuno-sqlgen imported successfully!")
print(f"Available type mappings: {list(TYPE_MAP.keys())}")

### Template Syntax Quick Reference

| Syntax | Description | Example |
|--------|-------------|---------|
| `@name:type` | Output column (result) | `@user_id:int` |
| `$name:type` | Escaped input variable | `$user_name:String` |
| `#name:type` | Literal input (raw SQL) | `#table_name` |
| `?name:type` | Option variable (metadata only) | `?batch_id:String` |
| `[...]` | Optional section | `[ WHERE x=$x ]` |
| `[AND]` / `[,]` | Combiner (included if previous was) | `[x=$x] [AND] [y=$y]` |
| `{}` | Stop-combiner (resets combine state) | `[a=$a] {} [AND b=$b]` |

Default type when `:type` is omitted is `String`.

---
## 2. Basic Examples

### 2.1 Set up a sample database

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("""
    CREATE TABLE users (
        user_id INTEGER PRIMARY KEY,
        user_name TEXT NOT NULL,
        email TEXT,
        birthdate TEXT,
        active INTEGER DEFAULT 1
    )
""")
conn.executemany(
    "INSERT INTO users (user_id, user_name, email, birthdate, active) VALUES (?, ?, ?, ?, ?)",
    [
        (1, "alice", "alice@example.com", "1990-01-15", 1),
        (2, "bob", "bob@example.com", "1985-06-20", 0),
        (3, "carol", "carol@example.com", "1992-11-03", 1),
        (4, "dave", "dave@example.com", "1988-03-25", 1),
    ],
)
conn.commit()
print("Database ready with 4 users.")

### 2.2 Ad-hoc API usage (inline templates)

Define SQL templates as strings and use them directly — no files needed.

In [ ]:
TEMPLATE = """
selectUser:=
    SELECT @user_id:int, @user_name:String, @email:String, @birthdate:Date
        FROM users
            [ WHERE
                [user_name=$user_name] [AND] [active=$active:int]
            ]
        ORDER BY user_id
        LIMIT 10;

insertUser:=
    INSERT INTO users (user_name, email, active) VALUES ($user_name, $email, $active:int);

updateUser:=
    UPDATE users SET
        [user_name=$user_name] [,]
        [email=$email] [,]
        [active=$active:int]
    WHERE user_id=$user_id:int;
"""

api = SQLGenApi(TEMPLATE)
print(f"Available statements: {api.statement_names}")

In [ ]:
# Query all users — no filters means no WHERE clause
df_all = api.select_user().query(conn)
print(f"All users ({len(df_all)} rows):")
df_all

In [ ]:
# Filter by name
api.select_user(user_name="alice").query(conn)

In [ ]:
# Multiple filters — AND combiner appears automatically
api.select_user(user_name="alice", active=1).query(conn)

In [ ]:
# Inspect the generated SQL
print("No filter:")
print(api.select_user().build_sql())

print("\nOne filter:")
print(api.select_user(user_name="alice").build_sql())

print("\nBoth filters:")
print(api.select_user(user_name="alice", active=1).build_sql())

In [ ]:
# INSERT a new user
rows = api.insert_user(user_name="eve", email="eve@example.com", active=1).execute(conn)
conn.commit()
print(f"Inserted {rows} row(s)")

# Verify
api.select_user().query(conn)

In [ ]:
# UPDATE only the fields you provide — comma combiners handle placement
api.update_user(user_id=1, email="alice_new@example.com").execute(conn)
conn.commit()

print("Update email only — SQL:")
print(api.update_user(user_id=1, email="alice_new@example.com").build_sql())

print("\nUpdate name + active — SQL:")
print(api.update_user(user_id=2, user_name="bob_updated", active=1).build_sql())

### 2.3 Code generation

Generate Python source files from templates. The generated code contains typed `@dataclass` classes with `build_sql()`, `query()`, and `execute()` methods.

In [ ]:
gen = PythonCodeGenerator()
code = gen.generate(TEMPLATE, "user_queries")

print(f"Generated {len(code)} chars of Python code")
print(code[:600])
print("...")

In [ ]:
# The generated code is valid Python — compile and use it
compile(code, "<generated>", "exec")
print("Generated code compiles successfully!")

# Execute it and use the factory functions
ns = {}
exec(code, ns)

params = ns["selectUser"](user_name="alice")
print(f"\nGenerated SQL: {params.build_sql().strip()[:100]}...")

---
## 3. Escaping Special Characters

The characters `$`, `#`, `@`, `?`, `[`, `]`, `{`, and `}` have special meaning in tamuno-sqlgen templates. To include them as literal text in the generated SQL, prefix them with a backslash (`\\`).

This is essential when writing templates for SQL dialects that use these characters natively — for example:

| Dialect | Characters | Use case |
|---------|-----------|----------|
| **PostgreSQL** | `$1`, `$2` | Positional parameters |
| **PostgreSQL** | `@>`, `?`, `?\|`, `?&` | JSONB operators |
| **SQL Server** | `[schema].[table]` | Bracket-quoted identifiers |
| **MySQL** | `@variable` | User-defined variables |

### 3.1 Escaping each special character

Use `\\` before any special character to produce the literal character in the output.

In [ ]:
# Escaping special characters with backslash
from tamuno_sqlgen import SQLGenApi

# Each special character can be escaped
escape_examples = {
    r'\$':  'literal dollar sign',
    r'\#':  'literal hash',
    r'\@':  'literal at-sign',
    r'\?':  'literal question mark',
    r'\[':  'literal open bracket',
    r'\]':  'literal close bracket',
    r'\{':  'literal open brace',
    r'\}':  'literal close brace',
}

for escaped, desc in escape_examples.items():
    template = f'test:= SELECT {escaped};'
    api = SQLGenApi(template)
    sql = api.test().build_sql().strip()
    print(f'{escaped:4s} -> {sql:30s}  ({desc})')

### 3.2 Escaping the backslash itself

Since `\\` is the escape character, use `\\\\` to produce a literal backslash in the output. You can also combine an escaped backslash with a variable — `\\\\$var` produces a literal `\\` followed by the variable's value.

In [ ]:
# Double backslash produces a literal backslash
api = SQLGenApi(r'test:= SELECT \\;')
print(r'\\           ->', api.test().build_sql().strip())

# Escaped backslash followed by a variable: literal \ + variable value
api = SQLGenApi(r'test:= SELECT \\$name;')
print(r'\\$name      ->', api.test(name='world').build_sql().strip())

# Escaped backslash + escaped dollar: literal \$
api = SQLGenApi(r'test:= SELECT \\\$name;')
print(r'\\\$name    ->', api.test().build_sql().strip())

### 3.3 Backslash inside quoted strings

Inside single or double quotes, the backslash is **not** an escape character — both the backslash and the following character are preserved as-is. This means SQL string literals with backslashes work naturally.

In [ ]:
# Backslash inside quotes is preserved (not treated as escape)
api = SQLGenApi(r"test:= SELECT '\n' as newline;")
print('Inside single quotes:', api.test().build_sql().strip())

api = SQLGenApi('test:= SELECT "path\\to\\file" as path;')
print('Inside double quotes:', api.test().build_sql().strip())

### 3.4 Practical examples: SQL dialect syntax

Templates that generate SQL for specific database features — showing how escaping lets you use tamuno-sqlgen with any dialect.

In [ ]:
# PostgreSQL: positional parameters and JSONB operators
pg_template = r"""
pgPositional:=
    SELECT * FROM events WHERE user_id = \$1 AND status = \$2;

pgJsonb:=
    SELECT * FROM documents
        WHERE metadata \@> $filter_json
        [ AND metadata \? $key_check ];
"""

pg_api = SQLGenApi(pg_template)

print('PostgreSQL positional parameters:')
print(pg_api.pg_positional().build_sql())

print('PostgreSQL JSONB contains + key check:')
print(pg_api.pg_jsonb(filter_json='{"type": "alert"}', key_check='priority').build_sql())

print('PostgreSQL JSONB contains only (no key check):')
print(pg_api.pg_jsonb(filter_json='{"type": "alert"}').build_sql())

In [ ]:
# SQL Server: bracket-quoted identifiers
mssql_template = r"""
mssqlQuery:=
    SELECT \[id\], \[user name\], \[email\]
        FROM \[dbo\].\[user table\]
        [ WHERE \[user name\] = $name ];
"""

mssql_api = SQLGenApi(mssql_template)

print('SQL Server bracket-quoted identifiers:')
print(mssql_api.mssql_query(name='alice').build_sql())

print('Without filter:')
print(mssql_api.mssql_query().build_sql())

In [ ]:
# MySQL: user-defined variables
mysql_template = r"""
mysqlVars:=
    SET \@row_num = 0;

mysqlRanked:=
    SELECT \@row_num := \@row_num + 1 AS rank, name
        FROM #table_name
        ORDER BY score DESC;
"""

mysql_api = SQLGenApi(mysql_template)

print('MySQL user variable initialization:')
print(mysql_api.mysql_vars().build_sql())

print('MySQL ranking with user variable:')
print(mysql_api.mysql_ranked(table_name='students').build_sql())

### 3.5 Escape rules summary

| Template | Output | Notes |
|----------|--------|-------|
| `\\$` | `$` | Literal dollar sign |
| `\\#` | `#` | Literal hash |
| `\\@` | `@` | Literal at-sign |
| `\\?` | `?` | Literal question mark |
| `\\[` | `[` | Literal bracket |
| `\\]` | `]` | Literal bracket |
| `\\{` | `{` | Literal brace |
| `\\}` | `}` | Literal brace |
| `\\\\` | `\\` | Literal backslash |
| `\\\\$var` | `\\` + value of `$var` | Escaped backslash then variable |
| Inside `'...'` | `\\` preserved | Not an escape character inside quotes |

---
## 4. Advanced SQL Examples

### 4.1 Set up an e-commerce database

In [ ]:
conn2 = sqlite3.connect(":memory:")
conn2.executescript("""
    CREATE TABLE customers (
        customer_id INTEGER PRIMARY KEY,
        customer_name TEXT NOT NULL,
        email TEXT,
        city TEXT
    );
    CREATE TABLE products (
        product_id INTEGER PRIMARY KEY,
        product_name TEXT NOT NULL,
        category TEXT,
        price REAL,
        stock INTEGER DEFAULT 0,
        description TEXT
    );
    CREATE TABLE orders (
        order_id INTEGER PRIMARY KEY,
        customer_id INTEGER REFERENCES customers(customer_id),
        order_date TEXT,
        status TEXT DEFAULT 'pending'
    );
    CREATE TABLE order_items (
        item_id INTEGER PRIMARY KEY,
        order_id INTEGER REFERENCES orders(order_id),
        product_id INTEGER REFERENCES products(product_id),
        quantity INTEGER,
        unit_price REAL
    );

    INSERT INTO customers VALUES
        (1, 'Alice', 'alice@example.com', 'New York'),
        (2, 'Bob', 'bob@example.com', 'London'),
        (3, 'Carol', 'carol@example.com', 'Paris');

    INSERT INTO products VALUES
        (1, 'Laptop', 'Electronics', 999.99, 50, 'High-performance laptop'),
        (2, 'Mouse', 'Electronics', 29.99, 200, 'Wireless mouse'),
        (3, 'Desk', 'Furniture', 349.00, 30, 'Standing desk'),
        (4, 'Chair', 'Furniture', 199.00, 45, 'Ergonomic chair'),
        (5, 'Notebook', 'Stationery', 4.99, 500, 'Lined notebook'),
        (6, 'Pen Set', 'Stationery', 12.99, 300, 'Premium pen set');

    INSERT INTO orders VALUES
        (1, 1, '2024-01-15', 'shipped'),
        (2, 1, '2024-03-20', 'shipped'),
        (3, 2, '2024-02-10', 'pending'),
        (4, 2, '2024-06-05', 'shipped'),
        (5, 3, '2024-04-12', 'cancelled'),
        (6, 3, '2024-07-01', 'shipped');

    INSERT INTO order_items VALUES
        (1, 1, 1, 1, 999.99),
        (2, 1, 2, 2, 29.99),
        (3, 2, 3, 1, 349.00),
        (4, 3, 4, 2, 199.00),
        (5, 4, 5, 10, 4.99),
        (6, 4, 6, 5, 12.99),
        (7, 5, 1, 1, 999.99),
        (8, 6, 2, 3, 29.99),
        (9, 6, 3, 1, 349.00);
""")
print("E-commerce database ready: customers, products, orders, order_items")

In [ ]:
ADVANCED_TEMPLATE = """
searchOrders:=
    SELECT o.order_id as @order_id:int,
           c.customer_name as @customer_name:String,
           p.product_name as @product_name:String,
           oi.quantity as @quantity:int,
           oi.unit_price as @unit_price:double,
           o.order_date as @order_date:Date
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        JOIN order_items oi ON o.order_id = oi.order_id
        JOIN products p ON oi.product_id = p.product_id
            [ WHERE
                [c.customer_name=$customer_name]
                [AND]
                [p.category=$category]
                [AND]
                [o.order_date >= $date_from:Date]
                [AND]
                [o.order_date <= $date_to:Date]
                [AND]
                [o.status=$status]
            ]
        ORDER BY o.order_date DESC;

salesReport:=
    SELECT p.category as @category:String,
           SUM(oi.quantity * oi.unit_price) as @total_revenue:double,
           COUNT(DISTINCT o.order_id) as @order_count:int,
           AVG(oi.unit_price) as @avg_price:double
        FROM products p
        JOIN order_items oi ON p.product_id = oi.product_id
        JOIN orders o ON oi.order_id = o.order_id
            [ WHERE
                [o.order_date >= $date_from:Date]
                [AND]
                [o.order_date <= $date_to:Date]
                [AND]
                [o.status=$status]
            ]
        GROUP BY p.category
            [ HAVING
                [SUM(oi.quantity * oi.unit_price) >= $min_revenue:double]
                [AND]
                [COUNT(DISTINCT o.order_id) >= $min_orders:int]
            ]
        ORDER BY total_revenue DESC;

paginatedProducts:=
    SELECT @product_id:int, @product_name:String, @category:String,
           @price:double, @stock:int
        FROM products
            [ WHERE
                [category=$category]
                [AND]
                [price >= $min_price:double]
                [AND]
                [price <= $max_price:double]
            ]
        ORDER BY #sort_column #sort_direction
        LIMIT #page_size OFFSET #page_offset;

updateProduct:=
    UPDATE products SET
        [product_name=$product_name] [,]
        [category=$category] [,]
        [price=$price:double] [,]
        [stock=$stock:int] [,]
        [description=$description]
    WHERE product_id=$product_id:int;

deleteOrders:=
    DELETE FROM orders
        [ WHERE
            [status=$status]
            [AND]
            [order_date < $before_date:Date]
            [AND]
            [customer_id=$customer_id:int]
        ];

customerSummary:=
    SELECT c.customer_name as @customer_name:String,
           COUNT(o.order_id) as @total_orders:int,
           COALESCE(SUM(oi.quantity * oi.unit_price), 0) as @total_spent:double
        FROM customers c
        LEFT JOIN orders o ON c.customer_id = o.customer_id
            [AND o.status=$status]
        LEFT JOIN order_items oi ON o.order_id = oi.order_id
        GROUP BY c.customer_name
        ORDER BY total_spent DESC;

insertProduct:=
    INSERT INTO products (product_name, category, price, stock, description)
        VALUES ($product_name, $category, $price:double, $stock:int, $description);

advancedSearch:=
    SELECT @id:int, @name:String, @category:String, @price:double
        FROM products
        [ WHERE
            [product_name LIKE $name_pattern]
            {}
            [AND category=$category]
            {}
            [AND price >= $min_price:double]
            [AND]
            [price <= $max_price:double]
        ];

selectFromDynamic:=
    SELECT @id:int, @name:String
        FROM #table_name
            [ WHERE
                [name=$name]
            ]
        ORDER BY id;
"""

api2 = SQLGenApi(ADVANCED_TEMPLATE)
print(f"Advanced statements: {api2.statement_names}")

### 4.2 Multi-table JOINs with dynamic filters

The `searchOrders` template joins 4 tables and has 5 optional WHERE filters. Any combination works — omitted filters are simply excluded from the SQL.

In [ ]:
# All orders — no WHERE clause
api2.search_orders().query(conn2)

In [ ]:
# Filter by customer
api2.search_orders(customer_name="Alice").query(conn2)

In [ ]:
from datetime import date

# Combine filters: shipped electronics in Q1 2024
api2.search_orders(
    category="Electronics",
    status="shipped",
    date_from=date(2024, 1, 1),
    date_to=date(2024, 3, 31),
).query(conn2)

In [ ]:
# Inspect the generated SQL
print("No filters — clean SQL without WHERE:")
print(api2.search_orders().build_sql())

print("\n" + "="*60)
print("\nWith filters — AND combiners placed automatically:")
print(api2.search_orders(customer_name="Alice", status="shipped").build_sql())

### 4.3 Aggregation with GROUP BY and optional HAVING

The `salesReport` template uses optional sections in both WHERE and HAVING, so you can filter and set thresholds dynamically.

In [ ]:
# Full report — no filters, no HAVING thresholds
api2.sales_report().query(conn2)

In [ ]:
# Shipped orders only, with revenue threshold
api2.sales_report(status="shipped", min_revenue=100.0).query(conn2)

In [ ]:
# Show the SQL with HAVING clause
print(api2.sales_report(
    status="shipped",
    min_revenue=100.0,
    min_orders=1,
).build_sql())

### 4.4 Pagination with literal variables

Literal variables (`#var`) inject raw values — perfect for column names, sort directions, and LIMIT/OFFSET parameters that should not be quoted.

In [ ]:
# Page 1: 3 items, sorted by price ascending
print("Page 1:")
display(api2.paginated_products(
    sort_column="price", sort_direction="ASC",
    page_size="3", page_offset="0",
).query(conn2))

# Page 2
print("\nPage 2:")
display(api2.paginated_products(
    sort_column="price", sort_direction="ASC",
    page_size="3", page_offset="3",
).query(conn2))

# With category filter
print("\nElectronics only:")
display(api2.paginated_products(
    category="Electronics",
    sort_column="product_name", sort_direction="DESC",
    page_size="10", page_offset="0",
).query(conn2))

### 4.5 Dynamic UPDATE with comma combiners

The `[,]` combiner only appears between fields that are actually included — so you can update any subset of fields without worrying about trailing commas.

In [ ]:
# Update just the price
print("Update price only:")
print(api2.update_product(product_id=1, price=899.99).build_sql())

print("\nUpdate name + price:")
print(api2.update_product(product_id=1, product_name="Laptop Pro", price=1099.99).build_sql())

print("\nUpdate all fields:")
print(api2.update_product(
    product_id=1, product_name="Laptop Pro", category="Computers",
    price=1099.99, stock=25, description="Updated laptop",
).build_sql())

In [ ]:
# Execute an update and verify
api2.update_product(product_id=2, price=24.99, stock=250).execute(conn2)
conn2.commit()

print("Products after update:")
api2.paginated_products(
    sort_column="product_id", sort_direction="ASC",
    page_size="10", page_offset="0",
).query(conn2)

### 4.6 Customer summary with LEFT JOIN

In [ ]:
# All customers with their order totals
print("All customers:")
display(api2.customer_summary().query(conn2))

# Shipped orders only
print("\nShipped orders only:")
display(api2.customer_summary(status="shipped").query(conn2))

### 4.7 DELETE with optional conditions

In [ ]:
print("DELETE SQL:")
print(api2.delete_orders(status="cancelled").build_sql())

# Execute
count_before = conn2.execute("SELECT COUNT(*) FROM orders").fetchone()[0]
api2.delete_orders(status="cancelled").execute(conn2)
conn2.commit()
count_after = conn2.execute("SELECT COUNT(*) FROM orders").fetchone()[0]
print(f"\nOrders: {count_before} -> {count_after} (deleted {count_before - count_after})")

### 4.8 Stop-combiners for independent filter groups

The `{}` stop-combiner resets the combiner state. This means each group of conditions is independent — an `AND` combiner only appears if its immediately preceding section (within the same group) was included.

In [ ]:
# Price range only
print("Price range only:")
print(api2.advanced_search(min_price=10.0, max_price=100.0).build_sql())

print("\nCategory only:")
print(api2.advanced_search(category="Electronics").build_sql())

print("\nAll filters:")
print(api2.advanced_search(
    name_pattern="%Lap%", category="Electronics",
    min_price=100.0, max_price=2000.0,
).build_sql())

### 4.9 Dynamic table names

In [ ]:
# Create a categories table for the dynamic query
conn2.execute("""
    CREATE TABLE IF NOT EXISTS categories AS
    SELECT DISTINCT
        ROW_NUMBER() OVER () as id,
        category as name
    FROM products
""")

# Query using a dynamic table name (literal variable)
print("Dynamic table 'categories':")
display(api2.select_from_dynamic(table_name="categories").query(conn2))

print("\nSQL with dynamic table + filter:")
print(api2.select_from_dynamic(table_name="categories", name="Electronics").build_sql())

### 4.10 SQL dialect differences

tamuno-sqlgen supports ANSI (default) and MySQL dialects for value escaping.

In [ ]:
ansi_api = SQLGenApi(TEMPLATE, dialect=SQLDialect())
mysql_api = SQLGenApi(TEMPLATE, dialect=MySQLDialect())

print("ANSI dialect (O'Brien -> doubled quotes):")
print(ansi_api.select_user(user_name="O'Brien").build_sql())

print("\nMySQL dialect (O'Brien -> backslash escape):")
print(mysql_api.select_user(user_name="O'Brien").build_sql())

---
## 5. Pandas Examples

tamuno-sqlgen's `.query()` method returns pandas DataFrames with columns named after the `@output` variables. This integrates naturally with the pandas ecosystem.

### 5.1 Column naming and dtypes

In [ ]:
import pandas as pd

df = api2.search_orders(status="shipped").query(conn2)

print(f"Column names (from @output vars): {list(df.columns)}")
print(f"\nColumn dtypes:")
print(df.dtypes)
print(f"\nShape: {df.shape}")
df

### 5.2 GroupBy, aggregation, and filtering

In [ ]:
# Group by customer
grouped = df.groupby("customer_name").agg(
    total_items=("quantity", "sum"),
    total_value=("unit_price", lambda x: (x * df.loc[x.index, "quantity"]).sum()),
    num_orders=("order_id", "nunique"),
)
print("Grouped by customer:")
grouped

In [ ]:
# Filter with pandas
electronics = df[df["product_name"].isin(["Laptop", "Mouse"])]
print(f"Electronics in shipped orders ({len(electronics)} items):")
electronics

### 5.3 Sales report with computed columns

In [ ]:
df_report = api2.sales_report(status="shipped").query(conn2)

# Add revenue percentage column
total = df_report["total_revenue"].sum()
df_report["pct_revenue"] = (df_report["total_revenue"] / total * 100).round(1)

print("Sales report with revenue percentages:")
df_report

### 5.4 Combining multiple query results

In [ ]:
df_products = api2.paginated_products(
    sort_column="product_id", sort_direction="ASC",
    page_size="100", page_offset="0",
).query(conn2)

df_summary = api2.customer_summary(status="shipped").query(conn2)

print(f"Products: {df_products.shape}")
display(df_products)

print(f"\nCustomer summary: {df_summary.shape}")
display(df_summary)

### 5.5 Exporting results

In [ ]:
# CSV
csv_output = df_products.to_csv(index=False)
print("CSV output:")
print(csv_output)

In [ ]:
# JSON
json_output = df_products.to_json(orient="records", indent=2)
print("JSON output:")
print(json_output)

In [ ]:
# List of dicts (useful for APIs)
records = df_products.to_dict(orient="records")
print(f"{len(records)} records:")
for row in records[:3]:
    print(f"  {row}")

---
## 6. Type-safe API Examples

tamuno-sqlgen can generate Python `@dataclass` classes for your templates, giving you IDE autocompletion, type checking, and type-safe result handling.

### 6.1 Code generation creates typed dataclasses

In [ ]:
gen = PythonCodeGenerator()
code = gen.generate(ADVANCED_TEMPLATE, "shop_queries")

# Show the generated Row and Params classes for searchOrders
lines = code.split("\n")
for i, line in enumerate(lines):
    if "class SearchOrdersRow:" in line or "class SearchOrdersParams:" in line:
        j = i
        while j < len(lines) and (
            j == i
            or (lines[j].startswith("    ") and not lines[j].strip().startswith("def "))
        ):
            print(lines[j])
            j += 1
        print("    ...")
        print()

### 6.2 Using generated Params dataclasses

In [ ]:
# Execute the generated code
ns = {}
exec(code, ns)

# Create typed params — IDE would show autocompletion here
params = ns["searchOrders"](customer_name="Alice", status="shipped")
print(f"Type: {type(params).__name__}")
print(f"Fields: {params}")
print(f"\nSQL:")
print(params.build_sql())

# Execute against database
print(f"\nResults:")
params.query(conn2)

### 6.3 Type-safe Row dataclasses

The generated code includes `Row` dataclasses that match the output columns. Convert DataFrame rows to typed objects for safe attribute access.

In [ ]:
SearchOrdersRow = ns["SearchOrdersRow"]
print(f"Row class: {SearchOrdersRow}")
print(f"Row fields: {[f for f in SearchOrdersRow.__dataclass_fields__]}")

# Convert DataFrame rows to typed objects
df = ns["searchOrders"](status="shipped").query(conn2)
rows = [SearchOrdersRow(**row) for row in df.to_dict(orient="records")]

print(f"\nTyped rows ({len(rows)}):")
for row in rows[:5]:
    print(f"  Order #{row.order_id}: {row.customer_name} bought "
          f"{row.quantity}x {row.product_name} @ ${row.unit_price}")

### 6.4 Custom typed result wrapper

Build your own typed wrappers around query results for domain-specific code.

In [ ]:
from dataclasses import dataclass

@dataclass
class Product:
    product_id: int
    product_name: str
    category: str
    price: float
    stock: int

def get_products(api, conn, page=1, page_size=10, **filters) -> list[Product]:
    """Fetch products with pagination and return typed Product objects."""
    offset = (page - 1) * page_size
    df = api.paginated_products(
        sort_column="product_id",
        sort_direction="ASC",
        page_size=str(page_size),
        page_offset=str(offset),
        **filters,
    ).query(conn)
    return [Product(**row) for row in df.to_dict(orient="records")]

products = get_products(api2, conn2)
print(f"Products ({len(products)} items):")
for p in products:
    print(f"  [{p.product_id}] {p.product_name} ({p.category}) - ${p.price:.2f}, stock: {p.stock}")

# Type-safe filtering in Python
cheap = [p for p in products if p.price < 50]
print(f"\nCheap products (< $50): {[p.product_name for p in cheap]}")

### 6.5 Factory introspection

Inspect a statement's input and output variables at runtime — useful for building dynamic UIs or validation.

In [ ]:
factory = api2.search_orders

print("Input parameters (what the query accepts):")
for var in factory.input_vars:
    py_type = TYPE_MAP.get(var.vartype, TYPE_MAP["String"])["python_type"]
    print(f"  ${var.value}: {var.vartype} -> Python {py_type.__name__}")

print(f"\nOutput columns (what the query returns):")
for var in factory.output_vars:
    py_type = TYPE_MAP.get(var.vartype, TYPE_MAP["String"])["python_type"]
    print(f"  @{var.value}: {var.vartype} -> Python {py_type.__name__}")

### 6.6 Type-safe updates with generated code

In [ ]:
# Full update
update_params = ns["updateProduct"](product_id=1, price=1199.99, stock=40)
print(f"Params: {update_params}")
print(f"SQL: {update_params.build_sql()}")

# Partial update — None fields are excluded from SQL
partial = ns["updateProduct"](product_id=1, price=799.99)
print(f"\nPartial params: {partial}")
print(f"Partial SQL: {partial.build_sql()}")

---
## 7. SQLAlchemy Examples

tamuno-sqlgen generates raw SQL strings. Combine them with SQLAlchemy's `text()`, connection management, and transactions.

### 7.1 Set up SQLAlchemy

In [ ]:
from sqlalchemy import create_engine, text

engine = create_engine("sqlite:///:memory:")

# Create schema and insert data
with engine.begin() as sa_conn:
    sa_conn.execute(text("""
        CREATE TABLE customers (
            customer_id INTEGER PRIMARY KEY,
            customer_name TEXT NOT NULL,
            email TEXT,
            city TEXT
        )
    """))
    sa_conn.execute(text("""
        CREATE TABLE products (
            product_id INTEGER PRIMARY KEY,
            product_name TEXT NOT NULL,
            category TEXT,
            price REAL,
            stock INTEGER DEFAULT 0,
            description TEXT
        )
    """))
    sa_conn.execute(text("""
        CREATE TABLE orders (
            order_id INTEGER PRIMARY KEY,
            customer_id INTEGER REFERENCES customers(customer_id),
            order_date TEXT,
            status TEXT DEFAULT 'pending'
        )
    """))
    sa_conn.execute(text("""
        CREATE TABLE order_items (
            item_id INTEGER PRIMARY KEY,
            order_id INTEGER REFERENCES orders(order_id),
            product_id INTEGER REFERENCES products(product_id),
            quantity INTEGER,
            unit_price REAL
        )
    """))
    sa_conn.execute(text("""
        INSERT INTO customers VALUES
            (1, 'Alice', 'alice@example.com', 'New York'),
            (2, 'Bob', 'bob@example.com', 'London'),
            (3, 'Carol', 'carol@example.com', 'Paris')
    """))
    sa_conn.execute(text("""
        INSERT INTO products VALUES
            (1, 'Laptop', 'Electronics', 999.99, 50, 'High-performance laptop'),
            (2, 'Mouse', 'Electronics', 29.99, 200, 'Wireless mouse'),
            (3, 'Desk', 'Furniture', 349.00, 30, 'Standing desk'),
            (4, 'Chair', 'Furniture', 199.00, 45, 'Ergonomic chair'),
            (5, 'Notebook', 'Stationery', 4.99, 500, 'Lined notebook'),
            (6, 'Pen Set', 'Stationery', 12.99, 300, 'Premium pen set')
    """))
    sa_conn.execute(text("""
        INSERT INTO orders VALUES
            (1, 1, '2024-01-15', 'shipped'),
            (2, 1, '2024-03-20', 'shipped'),
            (3, 2, '2024-02-10', 'pending'),
            (4, 2, '2024-06-05', 'shipped'),
            (5, 3, '2024-04-12', 'cancelled'),
            (6, 3, '2024-07-01', 'shipped')
    """))
    sa_conn.execute(text("""
        INSERT INTO order_items VALUES
            (1, 1, 1, 1, 999.99),
            (2, 1, 2, 2, 29.99),
            (3, 2, 3, 1, 349.00),
            (4, 3, 4, 2, 199.00),
            (5, 4, 5, 10, 4.99),
            (6, 4, 6, 5, 12.99),
            (7, 5, 1, 1, 999.99),
            (8, 6, 2, 3, 29.99),
            (9, 6, 3, 1, 349.00)
    """))

sa_api = SQLGenApi(ADVANCED_TEMPLATE)
print("SQLAlchemy database ready!")

### 7.2 Build SQL with tamuno-sqlgen, execute with SQLAlchemy

In [ ]:
# Build SQL with tamuno-sqlgen
sql = sa_api.search_orders(customer_name="Alice", status="shipped").build_sql()

# Execute with SQLAlchemy
with engine.connect() as sa_conn:
    result = sa_conn.execute(text(sql))
    rows = result.fetchall()
    print(f"Results ({len(rows)} rows):")
    for row in rows:
        print(f"  {row}")

### 7.3 Pandas + SQLAlchemy

`pd.read_sql_query` accepts SQLAlchemy connections, so combining all three is straightforward.

In [ ]:
with engine.connect() as sa_conn:
    df = pd.read_sql_query(
        sa_api.search_orders(status="shipped").build_sql(),
        sa_conn,
    )

print(f"DataFrame via SQLAlchemy ({len(df)} rows):")
df

### 7.4 Transactions

Use `engine.begin()` for atomic operations. Everything commits on success or rolls back on error.

In [ ]:
# Insert a new product in a transaction
with engine.begin() as sa_conn:
    insert_sql = sa_api.insert_product(
        product_name="Headphones",
        category="Electronics",
        price=79.99,
        stock=100,
        description="Noise-cancelling headphones",
    ).build_sql()
    sa_conn.execute(text(insert_sql))
    print("Inserted product via transaction")

# Verify
with engine.connect() as sa_conn:
    df = pd.read_sql_query(
        sa_api.paginated_products(
            sort_column="product_id", sort_direction="ASC",
            page_size="100", page_offset="0",
        ).build_sql(),
        sa_conn,
    )

print(f"Products after insert ({len(df)} rows):")
df

In [ ]:
# Multi-statement transaction: update + delete atomically
with engine.begin() as sa_conn:
    # Update price
    update_sql = sa_api.update_product(product_id=2, price=19.99).build_sql()
    sa_conn.execute(text(update_sql))

    # Delete cancelled orders
    delete_sql = sa_api.delete_orders(status="cancelled").build_sql()
    sa_conn.execute(text(delete_sql))

    print("Transaction committed: updated price + deleted cancelled orders")

# Verify
with engine.connect() as sa_conn:
    count = sa_conn.execute(text("SELECT COUNT(*) FROM orders")).scalar()
    price = sa_conn.execute(text("SELECT price FROM products WHERE product_id=2")).scalar()
    print(f"Remaining orders: {count}")
    print(f"Mouse price: ${price}")

### 7.5 Service layer pattern

Combine tamuno-sqlgen templates with a SQLAlchemy-backed service class for clean separation of concerns.

In [ ]:
class OrderService:
    """Service layer combining tamuno-sqlgen + SQLAlchemy."""

    def __init__(self, engine, template):
        self.engine = engine
        self.api = SQLGenApi(template)

    def search(self, **filters):
        """Search orders with any combination of filters."""
        sql = self.api.search_orders(**filters).build_sql()
        with self.engine.connect() as conn:
            return pd.read_sql_query(sql, conn)

    def get_report(self, **filters):
        """Get sales report grouped by category."""
        sql = self.api.sales_report(**filters).build_sql()
        with self.engine.connect() as conn:
            return pd.read_sql_query(sql, conn)

    def get_products_page(self, page=1, page_size=20,
                          sort_by="product_name", sort_dir="ASC", **filters):
        """Get a paginated page of products."""
        offset = (page - 1) * page_size
        sql = self.api.paginated_products(
            sort_column=sort_by,
            sort_direction=sort_dir,
            page_size=str(page_size),
            page_offset=str(offset),
            **filters,
        ).build_sql()
        with self.engine.connect() as conn:
            return pd.read_sql_query(sql, conn)

    def update_product(self, product_id, **fields):
        """Update a product, setting only the provided fields."""
        sql = self.api.update_product(product_id=product_id, **fields).build_sql()
        with self.engine.begin() as conn:
            conn.execute(text(sql))


svc = OrderService(engine, ADVANCED_TEMPLATE)
print("OrderService created")

In [ ]:
# Search
print("Alice's orders:")
display(svc.search(customer_name="Alice"))

# Report
print("\nShipped sales report:")
display(svc.get_report(status="shipped"))

# Paginated products
print("\nTop 3 most expensive products:")
display(svc.get_products_page(page=1, page_size=3, sort_by="price", sort_dir="DESC"))

In [ ]:
# Update via service layer
svc.update_product(1, price=949.99)

with engine.connect() as sa_conn:
    result = sa_conn.execute(text("SELECT price FROM products WHERE product_id=1"))
    print(f"Laptop price after service update: ${result.scalar()}")

In [ ]:
# Cleanup
conn.close()
conn2.close()
engine.dispose()
print("All connections closed. Tutorial complete!")

---

## Summary

| Feature | How |
|---------|-----|
| **Ad-hoc queries** | `SQLGenApi(template_string)` |
| **File-based templates** | `SQLGenApi.from_file("queries.sqlg")` |
| **Code generation** | `PythonCodeGenerator().generate_file(...)` |
| **Escaping** | Prefix special chars with `\\` (e.g. `\\$`, `\\[`, `\\@`) |
| **Query execution** | `params.query(conn)` returns DataFrame |
| **DML execution** | `params.execute(conn)` returns row count |
| **SQL inspection** | `params.build_sql()` returns SQL string |
| **SQLAlchemy** | Build SQL with tamuno-sqlgen, execute with `text(sql)` |
| **Transactions** | Use `engine.begin()` with tamuno-sqlgen SQL |
| **Type safety** | Generated dataclasses with typed fields |
| **Introspection** | `factory.input_vars`, `factory.output_vars` |

For more details, see the [GitHub repository](https://github.com/klondenberg-bioptimus/tamuno-sqlgen).